# Glacial lake mapping from Sentinel-2

Full pipeline: labels, training tiles, U-Net training, test-set comparison with NDWI/MNDWI, and the Lende / upper Bhotekoshi monitoring.
Needs internet access (Sentinel-2 is read from the Planetary Computer) and a GPU for training.

In [ ]:
import os, subprocess, sys
if not os.path.exists("src"):
    if not os.path.exists("glacial-lake-mapping-sentinel2"):
        subprocess.run(["git", "clone", "https://github.com/subarna007/glacial-lake-mapping-sentinel2"], check=True)
    os.chdir("glacial-lake-mapping-sentinel2")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(os.getcwd())

In [ ]:
!python -m pytest -q tests

## 1. Hi-MAG lake outlines for 2017

In [ ]:
!python -m src.labels

## 2. Training tiles

About 1,300 tiles of 2.56 km. Each one is a median of the clear Sentinel-2 scenes from 15 Sep to 30 Nov 2017. Tiles already on disk are skipped, so this cell can be re-run if it stops.

In [ ]:
!python -m src.chips --workers 8

In [ ]:
import pandas as pd
tiles = pd.read_csv("data/chips/tiles.csv")
tiles.groupby(["split", "kind"]).file.count().unstack()

## 3. Train the U-Net

In [ ]:
!python -m src.train --epochs 40

In [ ]:
import json, matplotlib.pyplot as plt
h = pd.DataFrame(json.load(open("results/unet/history.json")))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(h.epoch, h.train_loss); ax[0].set_title("training loss")
ax[1].plot(h.epoch, h.val_iou); ax[1].set_title("validation IoU")
plt.savefig("results/fig_training.png", dpi=150, bbox_inches="tight")

## 4. Test strips: U-Net vs NDWI vs MNDWI

In [ ]:
!python -m src.evaluate

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/results.md").read()))
display(Image("results/fig_recall_by_size.png"))
display(Image("results/fig_examples.png"))

## 5. Lende and upper Bhotekoshi, 2017-2025, and the 2025 and 2026 floods

In [ ]:
!python -m src.monitor

In [ ]:
display(pd.read_csv("results/monitor/totals_by_year.csv"))
display(pd.read_csv("results/monitor/lakes_by_year.csv").head(15))
display(Image("results/monitor/fig_annual.png"))
display(pd.read_csv("results/monitor/event_scenes.csv"))

In [ ]:
import shutil
shutil.make_archive("results_archive", "zip", "results")
print("results_archive.zip")